# Exercícios pandas parte 2-1

## Breve descrição

Resolução dos exercícios do material de Pandas da Universidade dos Dados.

In [8]:
# Importando bibliotecas
import pandas as pd
import numpy as np

# Carregando datasets
bank = pd.read_csv('/content/bank.csv', sep=";")
bank['job'].unique()

array(['unemployed', 'services', 'management', 'blue-collar',
       'self-employed', 'technician', 'entrepreneur', 'admin.', 'student',
       'housemaid', 'retired', 'unknown'], dtype=object)

# Exercício 1

Olhando apenas para o grupo de divorciados, qual o percentual deles que é empreendedor (entrepreneur)? E estudantes?

In [3]:
# Filtrando pelos divorciados
divorciados = bank[bank['marital'] == 'divorced']

# Agrupando os divorciados por idade e trabalho
tabela = pd.pivot_table(divorciados,
                        values='age',
                        index='job',
                        aggfunc='count')

# Renomeei a coluna para quantidade
tabela.columns = ['quantidade']

# Criando uma nova coluna com percentual de divorciado em cada categoria
tabela['percentual'] = (tabela['quantidade'] / tabela['quantidade'].sum()) * 100

# Recriando os índices
resultado = tabela.reindex(['entrepreneur', 'student'], fill_value=0)
print(resultado)

              quantidade  percentual
job                                 
entrepreneur          16    3.030303
student                0    0.000000


## Interpretação

- Há 3.03% de empresários divorciados.
- Não há estudantes divorciados.

## Aprendizados

- Como pivotar tabelas.
- Como recriar índices.
- Como filtrar dados específicos de colunas.

# Exercício 2

O seu chefe decidiu criar duas categorias, Premium e Gold, para focar nos clientes mais promissores.
Para a categoria Premium, entrarão os que possuem saldo (balance) nos top 10%. Ou seja, o grupo acima do percentil 90.
Já o grupo Gold, ficarão os que possuem saldo nos top 20%, mas abaixo do grupo Premium. Ou seja,
do p80 até o último do p90. Verifique quantos clientes possuem no grupo Premium e no Gold.

In [13]:
p80 = bank['balance'].quantile(0.80) # Pegando 80% dos dados
p90 = bank['balance'].quantile(0.90) # Pegando 90% dos dados


# Filtrando os dados para encaixar na categoria Gold
bank.loc[(bank['balance'] >= p80) & (bank['balance'] < p90), 'categoria'] = 'Gold'
# Filtrando os dados para encaixar na categoria Premium
bank.loc[bank['balance'] >= p90, 'categoria'] = 'Premium'

# Mostrando os resultados
contagem = bank['categoria'].value_counts()
contagem

,count
categoria,
Outros,3616
Premium,454
Gold,451


## Interpretação

- Há no grupo premium 454 pessoas.
- No grupo gold há 451 pessoas.

## Aprendizados

- Usar loc para filtrar dados.
- Usar a função quantile para pegar dados.

# Exercício 3

Seu chefe também quer saber se existe diferença nos percentuais vistos no exercício (1) para o grupo Premium.
Verifique como a tabela cruzada da população se diferencia do grupo Premium.
Existe diferença? Qual sua hipótese para que exista essa diferença?

In [19]:
# Filtrando divorciados que se encaixam na categoria Premium
divorciados_premium = bank[bank['categoria'] == 'Premium']

# Pegando o total de divorciados Premium
total_premium = len(divorciados_premium)

# Pegando o número de empresários divorciados que estão na categoria Premium
qtd_entrepreneur_premium = len(divorciados_premium[divorciados_premium['job'] == 'entrepreneur'])

# Pegando o número de estudantes divorciados que estão na categoria Premium
qtd_student_premium = len(divorciados_premium[divorciados_premium['job'] == 'student'])

# Calculando a porcentagem de empresários e estudantes divorciados Premium
pct_entrepreneur_premium = (qtd_entrepreneur_premium / total_premium) * 100 if total_premium > 0 else 0
pct_student_premium = (qtd_student_premium / total_premium) * 100 if total_premium > 0 else 0

print(f"Enpresários: {pct_entrepreneur_premium:.2f}%")
print(f"Estudantes: {pct_student_premium:.2f}%")

Enpresários: 3.52%
Estudantes: 1.98%


# Exercício 4

Para a apresentação final, você decide criar uma tabela com os 10 clientes que possuem maior saldo (balance).
Como a tabela é grande demais, você resolve apresentar apenas as informações relevantes:
age, job, marital, education, default, loan, duration.
Além disso, você decide destacar clientes que foram mais e menos atenciosos, pois são os que parecem ter maior e menor potencial,
respectivamente. Para isso, destaque na tabela, de amarelo, o cliente de maior duration.
Destaque, de vermelho, o que tem menor duration.

In [18]:
# Seleciona do DataFrame 'bank' apenas as linhas em que a coluna 'balance' é maior ou igual ao valor do percentil 90 (p90)
tabela10 = bank.loc[bank['balance'] >= p90]

# Seleciona apenas as 10 primeiras linhas do DataFrame filtrado
tabela10 = tabela10.head(10)

# Mantém apenas as colunas de interesse no DataFrame
tabela10 = tabela10[['age', 'job', 'marital', 'education', 'default', 'loan', 'duration']]

# Armazena o maior valor da coluna 'duration'
max_dur = tabela10['duration'].max()

# Armazena o menor valor da coluna 'duration'
min_dur = tabela10['duration'].min()

# Define uma função para aplicar estilos condicionais à coluna 'duration'
def destacar_duration_coluna(col):
    return ['background-color: yellow' if val == max_dur # Amarelo para o maior valor
            else 'background-color: red' if val == min_dur # Vermelho para o menor valor
            else '' for val in col]

# Aplica a função de destaque apenas na coluna 'duration' da tabela10, utilizando o Styler do pandas
tabela10_style = tabela10.style.apply(destacar_duration_coluna, subset=['duration'])

tabela10_style


,age,job,marital,education,default,loan,duration
1,33,services,married,secondary,no,yes,220
10,39,services,married,secondary,no,no,273
16,56,technician,married,secondary,no,no,239
25,41,management,married,tertiary,no,no,182
30,68,retired,divorced,secondary,no,no,897
38,33,management,married,secondary,no,no,765
40,38,management,single,tertiary,no,no,609
49,61,admin.,married,unknown,no,no,181
62,48,management,married,tertiary,no,no,227
64,56,admin.,married,secondary,no,no,223


## Aprendizados

- Como estilizar DataFrames

# Exercício 5

Na sua opinião e de parte da diretoria, há 3 fatores que determinam se a pessoa vai aceitar a proposta do seu banco (y):
o saldo (balance), se ela já deixou de pagar um empréstimo (default) e se ela tem um empréstimo pessoal (loan).
Para avaliar essas 3 características junto com o aceite (y), você precisará criar uma tabela dinâmica,
cujo valor observado seja a média do saldo. Avalie se existe diferença no saldo de quem aceita a proposta e
tem empréstimo vs quem não aceita e não tem empréstimo, e o mesmo para o default.
Em outras palavras, crie uma tabela dinâmica cruzando loan e default, mas também segmentando por aceite ou não da proposta.
O saldo de quem aceita e tem empréstimo é maior ou menor do que o de quem não aceita?
Por qual motivo você acha que isso acontece? Crie a sua hipótese para justificar o resultado achado!

In [7]:
bank.columns
bank.pivot_table(index=['default', 'loan'], columns='y', values='balance', aggfunc=np.mean)

/tmp/ipython-input-7-2875087265.py:2: FutureWarning: The provided callable <function mean at 0x7b926ea6a340> is currently using DataFrameGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  bank.pivot_table(index=['default', 'loan'], columns='y', values='balance', aggfunc=np.mean)


y                      no          yes
default loan                          
no      no    1516.932588  1667.687898
        yes    978.337600   798.634146
yes     no    -132.909091    61.428571
        yes   -468.652174   167.000000

## Possível interpretação

- Pessoas com maior saldo mesmo tendo empréstimo podem se sentir mais seguras financeiramente e, por isso, estão mais propensas a aceitar novas ofertas do banco.
- Já quem tem empréstimo e baixo saldo, talvez evite assumir novos compromissos, por receio de endividamento excessivo.
- No caso do default, inadimplentes com baixo saldo tendem a rejeitar ofertas — o banco deveria, talvez, não priorizar esse grupo.

# Exercício 6

Usando a tabela criada no exercício (5), qual o quadrante que possui menor saldo dentre os 8 da tabela dinâmica.
Por qual motivo você acredita que o saldo dele é o menor de todos?

## Resposta

1. Qual o quadrante com menor saldo?

  Resposta:

  default = yes

  loan = yes

  y = no

  Saldo médio: -468.65

2. Por que você acredita que esse grupo tem o menor saldo?
  </br>Resposta:
  - Já foram inadimplentes (default = yes), ou seja, têm histórico negativo de crédito;
  - Possuem um empréstimo pessoal ativo (loan = yes);
  - Não aceitaram a nova proposta do banco (y = no);
  - Têm saldo médio negativo, o que significa que estão em situação financeira crítica — devendo mais do que possuem no banco.